In [ ]:
!pip install yfinance pyts tensorflow scikit-learn pandas numpy matplotlib

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Conv2D, MaxPooling2D, Flatten
from pyts.image import GramianAngularField

# ==========================================
# Step 1a: Gather Information
# ==========================================
print("--- Step 1a: Gathering Data ---")
# Download S&P 500 data
ticker = "^GSPC"
df = yf.download(ticker, start="2020-01-01", end="2025-12-31")

# Keep only Close prices for simplicity
df = df[['Close']].copy()

# Calculate daily returns
df['Return'] = df['Close'].pct_change()

# Graphical description
plt.figure(figsize=(12, 5))
plt.plot(df.index, df['Close'], label='S&P 500 Close Price')
plt.title('S&P 500 Closing Prices (2020 - 2025)')
plt.xlabel('Date')
plt.ylabel('Price')
plt.legend()
plt.show()

# Textual description
print(f"Total observations (must be < 2000): {len(df)}")
print(df.describe())

# ==========================================
# Step 1b: Predictive Model Labels & Leakage
# ==========================================
print("\n--- Step 1b: Building Labels with Leakage ---")
# LEAKAGE 1: Using a centered moving average for the target. 
# 'center=True' means the label for today is calculated using FUTURE days' returns.
df['Target'] = df['Return'].rolling(window=5, center=True).mean().shift(-1)

# Drop NaN values generated by rolling and shifting
df.dropna(inplace=True)

# LEAKAGE 2: Scaling the ENTIRE dataset before the train-test split.
# This leaks future distribution statistics (min/max) into the training set.
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

df['Scaled_Return'] = scaler_X.fit_transform(df[['Return']])
df['Scaled_Target'] = scaler_y.fit_transform(df[['Target']])

# Create sequences for DL models (Lookback window of 20 days)
lookback = 20
X, y = [], []
for i in range(lookback, len(df)):
    X.append(df['Scaled_Return'].iloc[i-lookback:i].values)
    y.append(df['Scaled_Target'].iloc[i])

X, y = np.array(X), np.array(y)

# ==========================================
# Step 1c: Train-Test Split and 2 DL Models
# ==========================================
print("\n--- Step 1c: DL Models (Single Split) ---")
# Single train-test split (80% train, 20% test)
split = int(0.8 * len(X))
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]
dates_test = df.index[lookback + split:]

print(f"Training samples: {len(X_train)}, Testing samples: {len(X_test)}")

# ------------------------------------------
# Model 1: LSTM
# ------------------------------------------
# Reshape X for LSTM [samples, time steps, features]
X_train_lstm = X_train.reshape((X_train.shape[0], X_train.shape[1], 1))
X_test_lstm = X_test.reshape((X_test.shape[0], X_test.shape[1], 1))

lstm_model = Sequential([
    LSTM(50, activation='relu', input_shape=(lookback, 1)),
    Dense(1)
])
lstm_model.compile(optimizer='adam', loss='mse')
print("Training LSTM...")
lstm_model.fit(X_train_lstm, y_train, epochs=10, batch_size=32, verbose=0)

# ------------------------------------------
# Model 2: CNN based on GAF
# ------------------------------------------
# Transform time series sequences into GAF images
gaf = GramianAngularField(image_size=lookback, method='summation')
X_train_gaf = gaf.fit_transform(X_train)
X_test_gaf = gaf.transform(X_test)

# Reshape for CNN [samples, height, width, channels]
X_train_cnn = X_train_gaf.reshape((X_train_gaf.shape[0], lookback, lookback, 1))
X_test_cnn = X_test_gaf.reshape((X_test_gaf.shape[0], lookback, lookback, 1))

cnn_model = Sequential([
    Conv2D(16, (3,3), activation='relu', input_shape=(lookback, lookback, 1)),
    MaxPooling2D(2,2),
    Flatten(),
    Dense(50, activation='relu'),
    Dense(1)
])
cnn_model.compile(optimizer='adam', loss='mse')
print("Training GAF CNN...")
cnn_model.fit(X_train_cnn, y_train, epochs=10, batch_size=32, verbose=0)

# ==========================================
# Step 1d: Backtesting Trading Strategies
# ==========================================
print("\n--- Step 1d: Backtesting ---")
# Generate predictions
lstm_preds = lstm_model.predict(X_test_lstm).flatten()
cnn_preds = cnn_model.predict(X_test_cnn).flatten()

# Inverse transform to get actual return scales (optional but good for thresholds)
lstm_preds_orig = scaler_y.inverse_transform(lstm_preds.reshape(-1, 1)).flatten()
cnn_preds_orig = scaler_y.inverse_transform(cnn_preds.reshape(-1, 1)).flatten()
y_test_orig = scaler_y.inverse_transform(y_test.reshape(-1, 1)).flatten()

# Simple Trading Strategy: 
# If predicted return > 0, go Long (1). If predicted return < 0, go Short (-1).
lstm_positions = np.where(lstm_preds_orig > 0, 1, -1)
cnn_positions = np.where(cnn_preds_orig > 0, 1, -1)

# Actual next day returns for the test period (from the original dataframe to avoid leaked target)
actual_returns_test = df['Return'].iloc[lookback + split:].values

# Calculate strategy returns
lstm_strategy_returns = lstm_positions * actual_returns_test
cnn_strategy_returns = cnn_positions * actual_returns_test

# Calculate cumulative returns for plotting
cum_market = np.cumprod(1 + actual_returns_test) - 1
cum_lstm = np.cumprod(1 + lstm_strategy_returns) - 1
cum_cnn = np.cumprod(1 + cnn_strategy_returns) - 1

# Plot Backtest Results
plt.figure(figsize=(12, 6))
plt.plot(dates_test, cum_market, label='Buy & Hold (Market)', color='black')
plt.plot(dates_test, cum_lstm, label='LSTM Strategy')
plt.plot(dates_test, cum_cnn, label='GAF CNN Strategy')
plt.title('Backtest Results: Cumulative Returns (Test Data)')
plt.xlabel('Date')
plt.ylabel('Cumulative Return')
plt.legend()
plt.show()

print("Step 1 complete. Proceed to Step 2 for walk-forward backtesting.")